<a href="https://colab.research.google.com/github/Airelle-ws/Team-Bangan/blob/main/CSST102_Group6_CALABARZON_AnimalBite_ML_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSST102: Predictive Modeling and Risk Classification of Animal Bite Incidents in CALABARZON Using Supervised Machine Learning Algorithms
**Group 6 | BSCS 3-A | Laguna State Polytechnic University (LSPU)**  
**Members:** John Kurt S. Larioza (Leader), Airelle Lambert E. Ramos, Charles Lawrence R. Ferrer  
**UN SDG Alignment:** SDG 3 – Good Health and Well-being  
**DOST CALABARZON Hack4aProgress Track:** Track 3 – AI for Disaster & Health Resilience

In [ ]:
# Step 1: Import Libraries & Load CALABARZON Animal Bite Dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

df = pd.read_csv('calabarzon_animal_bite_2025_2026.csv')
print('Raw Dataset Shape:', df.shape)
df.head()

In [ ]:
# Step 2: Data Inspection, Cleaning (RA 10173 Compliance) & Feature Engineering
print('Missing Values Before Cleaning:\n', df.isnull().sum()[df.isnull().sum() > 0])
print('Duplicate Rows:', df.duplicated().sum())

df_clean = df.drop_duplicates().drop(columns=['Registration_No']).copy()
df_clean['Wound_Washed'] = df_clean['Wound_Washed'].fillna(df_clean['Wound_Washed'].mode()[0])
df_clean['Animal_Ownership'] = df_clean['Animal_Ownership'].fillna(df_clean['Animal_Ownership'].mode()[0])

# Domain-Specific Feature Engineering (DOH NRPCP Guidelines)
df_clean['High_Risk_Site'] = df_clean['Bite_Site'].isin(['Head/Neck', 'Hands/Fingers']).astype(int)
df_clean['Is_Pediatric'] = (df_clean['Age'] < 15).astype(int)

print('Cleaned Dataset Shape:', df_clean.shape)
print('\nExposure Category Distribution:\n', df_clean['Exposure_Category'].value_counts())

In [ ]:
# Step 3: Encoding & Stratified 80/20 Train-Test Split
X = df_clean.drop(columns=['Exposure_Category'])
y = df_clean['Exposure_Category']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

categorical_cols = [
    'Province', 'Municipality_City', 'Sex', 'Animal_Ownership', 'Animal_Type',
    'Bite_Site', 'Wound_Type', 'Wound_Washed', 'Previously_Immunized', 'Animal_Status_14d'
]
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)
print(f'Train Samples: {X_train.shape[0]} | Test Samples: {X_test.shape[0]} | Total Encoded Features: {X_encoded.shape[1]}')

In [ ]:
# Step 4: Baseline (Logistic Regression), Comparison (Random Forest & GB), and Tuned Gradient Boosting
lr_model = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
rf_base = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
gb_base = GradientBoostingClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

sample_weights = compute_sample_weight('balanced', y_train)
gb_tuned = GradientBoostingClassifier(n_estimators=90, max_depth=3, learning_rate=0.05, random_state=42)
gb_tuned.fit(X_train, y_train, sample_weight=sample_weights)

models = [
    ('Logistic Regression (Baseline)', lr_model.predict(X_test)),
    ('Random Forest (Default)', rf_base.predict(X_test)),
    ('Gradient Boosting (Default)', gb_base.predict(X_test)),
    ('Gradient Boosting (Tuned + Balanced)', gb_tuned.predict(X_test))
]

summary = []
for name, preds in models:
    summary.append({
        'Model': name,
        'Accuracy (%)': round(accuracy_score(y_test, preds)*100, 2),
        'Macro Precision (%)': round(precision_score(y_test, preds, average='macro', zero_division=0)*100, 2),
        'Macro Recall (%)': round(recall_score(y_test, preds, average='macro', zero_division=0)*100, 2),
        'Macro F1-Score (%)': round(f1_score(y_test, preds, average='macro', zero_division=0)*100, 2)
    })

display(pd.DataFrame(summary))
print('\nFinal Classification Report (Tuned Gradient Boosting):\n')
print(classification_report(y_test, models[-1][1], target_names=le.classes_))